In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:

import requests
import pandas as pd

from datetime import datetime, timezone
from zoneinfo import ZoneInfo
from google.cloud import bigquery

# Configurações
PROJECT_ID = "macrodata-brasil"

TABELA_BRONZE = (
    f"{PROJECT_ID}.bronze.ibge_ipca_raw"
)

URL_BASE = (
    "https://apisidra.ibge.gov.br"
    "/values/t/1737/n1/all/v/63"
)

MESES_REPROCESSAMENTO = 3
MESES_CARGA_INICIAL = 36

# Cliente BigQuery
client = bigquery.Client(
    project=PROJECT_ID,
    location="southamerica-east1"
)

# 1. Consultar última competência da Bronze
consulta = f"""
SELECT
    MAX(
        SAFE.PARSE_DATE('%Y%m', periodo)
    ) AS ultimo_periodo
FROM `{TABELA_BRONZE}`
"""

resultado = client.query(consulta).result()

ultimo_periodo = next(iter(resultado)).ultimo_periodo

# 2. Definir janela de extração
hoje = datetime.now(
    ZoneInfo("America/Sao_Paulo")
).date()

if ultimo_periodo is None:
    quantidade_meses = MESES_CARGA_INICIAL

else:
    meses_decorridos = (
        (hoje.year - ultimo_periodo.year) * 12
        + hoje.month - ultimo_periodo.month
    )

    quantidade_meses = (
        max(0, meses_decorridos)
        + MESES_REPROCESSAMENTO
        + 1
    )

# 3. Montar URL dinâmica
URL = f"{URL_BASE}/p/last%20{quantidade_meses}"

print("Última competência na Bronze:", ultimo_periodo)
print("Períodos solicitados:", quantidade_meses)
print("URL:", URL)

# 4. Consultar API IBGE/SIDRA
resposta = requests.get(
    URL,
    timeout=60
)

resposta.raise_for_status()

dados = resposta.json()

if not isinstance(dados, list) or len(dados) < 2:
    raise ValueError(
        "A API não retornou observações válidas."
    )

# 5. Converter para DataFrame
df_ipca = pd.DataFrame(dados[1:])

data_extracao = datetime.now(timezone.utc)

print("Registros extraídos:", len(df_ipca))
print("Data da extração:", data_extracao.isoformat())

display(df_ipca.tail())


Última competência na Bronze: 2026-10-01
Períodos solicitados: 5
URL: https://apisidra.ibge.gov.br/values/t/1737/n1/all/v/63/p/last%205
Registros extraídos: 5
Data da extração: 2026-11-17T12:51:37.197272+00:00


,NC,NN,MC,MN,V,D1C,D1N,D2C,D2N,D3C,D3N
0,1,Brasil,2,%,0.16,1,Brasil,63,IPCA - Variação mensal,202606,junho 2026
1,1,Brasil,2,%,0.07,1,Brasil,63,IPCA - Variação mensal,202607,julho 2026
2,1,Brasil,2,%,-0.32,1,Brasil,63,IPCA - Variação mensal,202608,agosto 2026
3,1,Brasil,2,%,0.82,1,Brasil,63,IPCA - Variação mensal,202609,setembro 2026
4,1,Brasil,2,%,0.14,1,Brasil,63,IPCA - Variação mensal,202610,outubro 2026


In [ ]:

import pandas as pd

# Mapeamento das colunas da API SIDRA
mapeamento_colunas = {
    "D1N": "localidade",
    "D2C": "codigo_indicador",
    "D2N": "indicador",
    "D3C": "periodo",
    "D3N": "descricao_periodo",
    "V": "valor",
    "MN": "unidade"
}

# Preparar os dados para a Bronze
df_ipca_staging = (
    df_ipca[list(mapeamento_colunas.keys())]
    .rename(columns=mapeamento_colunas)
    .copy()
)

# Adicionar metadados
df_ipca_staging["data_extracao"] = data_extracao
df_ipca_staging["fonte"] = "IBGE_SIDRA_1737_63"

# Preservar os campos originais como texto
colunas_texto = [
    "localidade",
    "codigo_indicador",
    "indicador",
    "periodo",
    "descricao_periodo",
    "valor",
    "unidade",
    "fonte"
]

for coluna in colunas_texto:
    df_ipca_staging[coluna] = (
        df_ipca_staging[coluna].astype(str)
    )

# Validar DataFrame vazio
if df_ipca_staging.empty:
    raise ValueError("Nenhum registro de IPCA disponível.")

# Validar competência mensal (AAAAMM)
periodos = pd.to_datetime(
    df_ipca_staging["periodo"],
    format="%Y%m",
    errors="coerce"
)

if periodos.isna().any():
    raise ValueError("Existem competências inválidas.")

# Validar valores numéricos
valores = pd.to_numeric(
    df_ipca_staging["valor"],
    errors="coerce"
)

if valores.isna().any():
    raise ValueError("Existem valores de IPCA inválidos.")

# Verificar competências duplicadas
if df_ipca_staging["periodo"].duplicated().any():
    raise ValueError("Existem competências duplicadas.")

# Organizar por período
df_ipca_staging = (
    df_ipca_staging
    .sort_values("periodo")
    .reset_index(drop=True)
)

print("Preparação concluída!")
print("Registros:", len(df_ipca_staging))
print("Primeiro período:", df_ipca_staging["periodo"].iloc[0])
print("Último período:", df_ipca_staging["periodo"].iloc[-1])

display(df_ipca_staging.head())


Preparação concluída!
Registros: 5
Primeiro período: 202606
Último período: 202610


,localidade,codigo_indicador,indicador,periodo,descricao_periodo,valor,unidade,data_extracao,fonte
0,Brasil,63,IPCA - Variação mensal,202606,junho 2026,0.16,%,2026-11-17 12:51:37.197272+00:00,IBGE_SIDRA_1737_63
1,Brasil,63,IPCA - Variação mensal,202607,julho 2026,0.07,%,2026-11-17 12:51:37.197272+00:00,IBGE_SIDRA_1737_63
2,Brasil,63,IPCA - Variação mensal,202608,agosto 2026,-0.32,%,2026-11-17 12:51:37.197272+00:00,IBGE_SIDRA_1737_63
3,Brasil,63,IPCA - Variação mensal,202609,setembro 2026,0.82,%,2026-11-17 12:51:37.197272+00:00,IBGE_SIDRA_1737_63
4,Brasil,63,IPCA - Variação mensal,202610,outubro 2026,0.14,%,2026-11-17 12:51:37.197272+00:00,IBGE_SIDRA_1737_63


In [ ]:

import io
import json
import pandas as pd
from google.cloud import bigquery

# Tabela intermediária
TABELA_STAGING = (
    "macrodata-brasil.bronze.ibge_ipca_staging"
)

# 1. Validar os dados preparados
if df_ipca_staging.empty:
    raise ValueError(
        "Nenhum registro de IPCA para carregar."
    )

registros = df_ipca_staging.copy()

# Validar as colunas esperadas
colunas_esperadas = [
    "localidade",
    "codigo_indicador",
    "indicador",
    "periodo",
    "descricao_periodo",
    "valor",
    "unidade",
    "data_extracao",
    "fonte"
]

if list(registros.columns) != colunas_esperadas:
    raise ValueError(
        "Estrutura de colunas inesperada."
    )

# 2. Formatar timestamp em UTC
registros["data_extracao"] = (
    pd.to_datetime(
        registros["data_extracao"],
        utc=True
    ).dt.strftime("%Y-%m-%dT%H:%M:%S.%fZ")
)

# 3. Converter DataFrame em JSONL
conteudo_jsonl = "\n".join(
    json.dumps(registro, ensure_ascii=False)
    for registro in registros.to_dict(orient="records")
)

arquivo_memoria = io.BytesIO(
    conteudo_jsonl.encode("utf-8")
)

# 4. Configurar carga no BigQuery
configuracao = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    schema=[
        bigquery.SchemaField("localidade", "STRING"),
        bigquery.SchemaField("codigo_indicador", "STRING"),
        bigquery.SchemaField("indicador", "STRING"),
        bigquery.SchemaField("periodo", "STRING"),
        bigquery.SchemaField("descricao_periodo", "STRING"),
        bigquery.SchemaField("valor", "STRING"),
        bigquery.SchemaField("unidade", "STRING"),
        bigquery.SchemaField("data_extracao", "TIMESTAMP"),
        bigquery.SchemaField("fonte", "STRING"),
    ],
)

# 5. Carregar somente na staging
job = client.load_table_from_file(
    arquivo_memoria,
    TABELA_STAGING,
    job_config=configuracao,
    location="southamerica-east1",
)

job.result()

# 6. Conferir quantidade carregada
tabela = client.get_table(TABELA_STAGING)

if tabela.num_rows != len(registros):
    raise ValueError(
        "Quantidade carregada diferente da extração."
    )

print("Staging do IPCA carregada com sucesso!")
print("Registros carregados:", tabela.num_rows)
print("Tabela:", TABELA_STAGING)
print("ID do job:", job.job_id)


Staging do IPCA carregada com sucesso!
Registros carregados: 5
Tabela: macrodata-brasil.bronze.ibge_ipca_staging
ID do job: acd6652d-3ea1-4b6e-a254-0fdd9e4cf3d1
